#### **About Dataset**
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

- asin - ID of the product, like B000FA64PK 
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary of the review (description).
- unixReviewTime - unix timestamp.


Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

**Inspiration**

- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What - factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis


**Best Practises**

- Preprocessing And Cleaning 
- Train Test Split
- BOW,TFIDF,Word2vec
- Train ML algorithms

In [1]:
# Load the dataset

import pandas as pd 
data = pd.read_csv('all_kindle_review.csv')

In [2]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [29]:
df = data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [30]:
df.shape

(12000, 2)

In [31]:
# Missing values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [32]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [33]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

#### Preprocessing and Cleaning

In [34]:
# Positive review is 1 and negative review is 0

df['rating'] = data['rating'].apply(lambda x:0 if x<3 else 1)

In [35]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [36]:
# 1. Lower all the cases

df['reviewText'] = df['reviewText'].str.lower()

In [37]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [38]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Shristi
[nltk_data]     Priya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### **BeautifulSoup**

**BeautifulSoup** is a Python library used to **parse and extract data from HTML and XML documents**.

It helps clean text by **removing HTML tags and extracting only the readable content**.

### Example

HTML text:

```
<p>This is a <b>great</b> product</p>
```

Using BeautifulSoup:

```
This is a great product

In [39]:
# 1. Remove special characters
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+', ' ', x))

# 2. Remove stopwords
df['reviewText'] = df['reviewText'].apply(
    lambda x: " ".join([y for y in x.split() if y not in stopwords.words('english')])
)

# 3. Remove URLs
df['reviewText'] = df['reviewText'].apply(
    lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '', str(x))
)

# 4. Remove HTML tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

# 5. Remove extra spaces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

In [40]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four books expecting 34 con...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [42]:
# Lemmatizer

from nltk.stem import WordNetLemmatizer

In [43]:
lemmatizer = WordNetLemmatizer()

In [44]:
def lemmatize_words(text):
    # Define a function that performs lemmatization on a given text

    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
    # text.split()
    # Split the sentence into individual words

    # lemmatizer.lemmatize(word)
    # Convert each word into its base (dictionary) form

    # [ ... for word in text.split() ]
    # Apply lemmatization to every word using list comprehension

    # " ".join(...)
    # Combine all processed words back into a single cleaned sentence

In [45]:
df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))
# Apply the lemmatize_words() function to every row in the 'reviewText' column
# .apply() processes each text entry one by one
# lambda x passes each review text to the lemmatize_words() function
# The result is stored back in the 'reviewText' column with words converted to their base form

In [46]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting 34 conc...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [48]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [51]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 24820))

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)

In [53]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [ ]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [ ]:
y_pred_tfidf = nb_model_bow.predict(X_test_tfidf)

In [56]:
confusion_matrix(y_test,y_pred_bow)

array([[534, 259],
       [763, 844]])

In [57]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.5741666666666667


In [58]:
confusion_matrix(y_test,y_pred_tfidf)

array([[517, 276],
       [760, 847]])

In [59]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5683333333333334
